In [1]:
import re
import json
import ast
import sys
sys.path.append(r"D:\Desktop_fake\MawileBot\home\MawileBot\src")
from poke_lib import most_similar

def parse_pokedex(raw):
    lines = raw.splitlines()
    lines = [line.strip() for line in lines if line.strip()]  # remove empty lines

    data = {}
    errors = []

    for i, line in enumerate(lines, 1):
        if line.endswith(","):
            line = line[:-1]
        try:
            key, value = line.split(":", 1)
            key = key.strip().strip('"')  # remove quotes around key
            value = ast.literal_eval(value.strip())  # safe eval of list/dict
            data[key] = value
        except Exception as e:
            errors.append(f"Line {i} parsing error: {e} → {line}")

    # ---------- CHECK ----------
    print(f"Parsed {len(data)} Pokémon")
    if errors:
        print("\nErrors:")
        for e in errors:
            print("-", e)

    new_data ={}
    exluded = ['yanmega','meganium','mrmimegalar']
    multievo = {'oddish': 'oddishvileplume',
                'gloom': 'gloomvileplume',
                'poliwag': 'poliwagpoliwrath',
                'poliwhirl': 'poliwhirlpoliwrath',
                "slowpoke" : "slowpokeslowbro",
                "slowpokegalar" : "slowpokeslowbrogalar"}
    
    renames = {}
    with open(r'D:\Desktop_fake\MawileBot\home\MawileBot\src\pokemon_list.json', 'r', encoding="utf-8") as f:
                choices = json.load(f)

    for pokemon, value in data.items():
        if ('mega' in pokemon and pokemon not in exluded) or pokemon == '':
            pass
        else:
            evo_info = []
            #print(pokemon)
            type = 'maybe_base'
            for values in data.values():
                # check if this Pokémon appears as a next evolution
                next_evo = values[4][0] if values[4] else None
                if pokemon == next_evo:
                    type = 'not_base'
                    break

            if type == 'not_base':
                if value[4] == []:
                    type = 'last'
                else:
                    type = 'mid'
            elif type == 'maybe_base' and value[4] != []:
                type = 'base'
            else:
                continue
            #print(type)
            evo_info.append(type)
            if type == 'base':
                evo_info.append([value[4][1],most_similar(value[4][0], choices, r = False)])
                if data[value[4][0]][4] != []:
                    evo_info.append([data[value[4][0]][4][1],most_similar(data[value[4][0]][4][0], choices, r = False)])
            elif type == 'mid':
                for key, values in data.items():
                    if pokemon in values[4]:
                        evo_info.append([values[4][1],most_similar(key, choices, r = False)])
                evo_info.append([value[4][1],most_similar(value[4][0], choices, r = False)])
            elif type == 'last':
                for key, values in data.items():
                    if pokemon in values[4]:
                        if values[1] != 'base':
                            for key2, values2 in data.items():
                                if key in values2[4]:
                                    evo_info.append([values2[4][1],most_similar(key2, choices, r = False)])
                        evo_info.append([values[4][1],most_similar(key, choices, r = False)])
        

            pokemon_check = most_similar(pokemon, choices, r = False)
            if pokemon_check != pokemon:
                renames[pokemon] = pokemon_check
                pokemon = pokemon_check           

            new_data[pokemon] = evo_info
        
    return new_data, errors, renames

In [2]:
with open(r'D:\Desktop_fake\MawileBot\home\utils\pokemon_evo_list\NewMegaScript.txt', 'r', encoding='utf-8') as f:
    raw = f.read()

pokedex, wrong, renames = parse_pokedex(raw)

# write out JSON
with open(r'D:\Desktop_fake\MawileBot\home\utils\pokemon_evo_list\evo_file.json', 'w', encoding='utf-8') as f:
    json.dump(pokedex, f, indent=4, ensure_ascii=False)

print(f"Wrote pokedex.json with {len(pokedex)} entries.")
if wrong:
    print("\nThe following lines were in the wrong format and skipped:")
    for ln in wrong:
        print("  •", ln)
if renames:
    print("\nThe following Pokémon names were renamed:")
    for old, new in renames.items():
        if 'galar' not in old and 'hisui' not in old and 'alola' not in old:
            print(f"  • {old} → {new}")

Parsed 1436 Pokémon

Errors:
- Line 1 parsing error: not enough values to unpack (expected 2, got 1) → {
- Line 1440 parsing error: not enough values to unpack (expected 2, got 1) → }
Wrote pokedex.json with 897 entries.

The following lines were in the wrong format and skipped:
  • Line 1 parsing error: not enough values to unpack (expected 2, got 1) → {
  • Line 1440 parsing error: not enough values to unpack (expected 2, got 1) → }

The following Pokémon names were renamed:
  • nidoranfemmina → nidoran-f
  • nidoranmaschio → nidoran-m
  • oddishvileplume → oddish
  • gloomvileplume → gloom
  • oddishbellossom → oddish
  • gloombellossom → gloom
  • poliwagpoliwrath → poliwag
  • poliwhirlpoliwrath → poliwhirl
  • poliwagpolitoed → poliwag
  • poliwhirlpolitoed → poliwhirl
  • slowpokeslowbro → slowpoke
  • slowpokeslowking → slowpoke
  • tyroguehitmonlee → tyrogue
  • tyroguehitmonchan → tyrogue
  • tyroguehitmontop → tyrogue
  • mimejr → mime-jr
  • mrmime → mr-mime
  • mrrime → mr